In [1]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider
print('Imports OK')

Imports OK


In [2]:
DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/pythonScripts/radarDataset/test_parallel"
print(f'Data directory: {DATA_DIR}')

Data directory: /home/tim-external/ros_ws/src/fsregistration/pythonScripts/radarDataset/test_parallel


In [3]:
import glob as glob_mod


def parse_results_csv(filepath):
    """Parse a results.csv file, returning metadata dict and DataFrame."""
    meta = {}
    lines = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('# '):
                key, val = line[2:].split(':', 1)
                val = val.strip()
                # Try to convert to numeric
                try:
                    if '.' in val or 'e' in val.lower():
                        val = float(val)
                    else:
                        val = int(val)
                except ValueError:
                    pass
                meta[key.strip()] = val
            elif not line.startswith('#'):
                lines.append(line)
    
    if not lines:
        return meta, None
    
    df = pd.read_csv(filepath, comment='#')
    return meta, df


# Scan for sequence directories
seq_dirs = sorted(glob_mod.glob(os.path.join(DATA_DIR, 'seq*_fs2d_*')))
print(f'Found {len(seq_dirs)} sequence directories:')

sequences = []
for seq_dir in seq_dirs:
    results_file = os.path.join(seq_dir, 'results.csv')
    if not os.path.exists(results_file):
        continue
    meta, df = parse_results_csv(results_file)
    if df is None or len(df) == 0:
        print(f'  {os.path.basename(seq_dir)}: empty or no data')
        continue
    seq_name = os.path.basename(seq_dir)
    sequences.append({
        'name': seq_name,
        'path': seq_dir,
        'df': df,
        'N': meta.get('N', 128),
        'size_of_pixel': meta.get('size_of_pixel', 0.5),
        'matching_step': meta.get('matching_step', 5),
        'total_frames': meta.get('total_frames', len(df) * 5 + 5),
        'num_pairs': len(df),
        'avg_rot_error': meta.get('avg_rot_error_deg', 0),
        'avg_trans_error': meta.get('avg_trans_error_m', 0),
        'avg_confidence': meta.get('avg_confidence', 0),
    })
    print(f'  {seq_name}: {len(df)} pairs, N={meta.get("N")}, pixel={meta.get("size_of_pixel")}m')

print(f'Total sequences loaded: {len(sequences)}')

Found 5 sequence directories:
  seq00_fs2d_N128_p200_s5: 828 pairs, N=128, pixel=2.0m
  seq00_fs2d_N128_p50_s5: 828 pairs, N=128, pixel=0.5m
  seq01_fs2d_N128_p50_s5: 769 pairs, N=128, pixel=0.5m
  seq05_fs2d_N128_p50_s5: 755 pairs, N=128, pixel=0.5m
Total sequences loaded: 4


In [4]:
def compute_trajectory(df, rot_col='est_rot_deg', tx_col='est_tx_m', ty_col='est_ty_m'):
    """Compute cumulative 2D trajectory from relative poses.
    
    Each row gives a relative pose: rotation (deg) and translation (m) in the
    current local frame. Compose them to get global position and heading.
    """
    x = [0.0]
    y = [0.0]
    heading = [0.0]  # degrees
    
    for _, row in df.iterrows():
        h_rad = np.radians(heading[-1])
        dx = row[tx_col]
        dy = row[ty_col]
        dtheta = row[rot_col]
        
        new_x = x[-1] + np.cos(h_rad) * dx - np.sin(h_rad) * dy
        new_y = y[-1] + np.sin(h_rad) * dx + np.cos(h_rad) * dy
        new_heading = heading[-1] + dtheta
        
        x.append(new_x)
        y.append(new_y)
        heading.append(new_heading)
    
    return np.array(x), np.array(y), np.array(heading)


# Compute trajectories for all sequences
trajectories = {}
for seq in sequences:
    df = seq['df']
    est_x, est_y, est_heading = compute_trajectory(df, 'est_rot_deg', 'est_tx_m', 'est_ty_m')
    gt_x, gt_y, gt_heading = compute_trajectory(df, 'gt_rot_deg', 'gt_tx_m', 'gt_ty_m')
    trajectories[seq['name']] = {
        'est_x': est_x,
        'est_y': est_y,
        'est_heading': est_heading,
        'gt_x': gt_x,
        'gt_y': gt_y,
        'gt_heading': gt_heading,
    }
    print(f"{seq['name']}: est trajectory length {len(est_x)-1} steps, span ({est_x.max()-est_x.min():.1f} x {est_y.max()-est_y.min():.1f}) m")

print('Trajectories computed.')

seq00_fs2d_N128_p200_s5: est trajectory length 828 steps, span (2735.3 x 1086.0) m
seq00_fs2d_N128_p50_s5: est trajectory length 828 steps, span (411.3 x 952.9) m
seq01_fs2d_N128_p50_s5: est trajectory length 769 steps, span (1721.5 x 504.6) m
seq05_fs2d_N128_p50_s5: est trajectory length 755 steps, span (997.1 x 1118.9) m
Trajectories computed.


In [5]:
if len(sequences) > 0:
    num_seqs = len(sequences)
    
    @interact(seq_idx=IntSlider(
        min=0,
        max=num_seqs - 1,
        value=0,
        step=1,
        description='Sequence',
        continuous_update=False,
        readout=True,
        readout_format='d'
    ))
    def plot_trajectory(seq_idx=0):
        seq = sequences[seq_idx]
        seq_name = seq['name']
        traj = trajectories[seq_name]
        
        est_x = traj['est_x']
        est_y = traj['est_y']
        gt_x = traj['gt_x']
        gt_y = traj['gt_y']
        
        fig = go.Figure()
        
        # Ground truth trajectory
        fig.add_trace(go.Scatter(
            x=gt_x,
            y=gt_y,
            mode='lines',
            name='Ground Truth',
            line=dict(color='green', width=2),
        ))
        
        # Estimated trajectory
        fig.add_trace(go.Scatter(
            x=est_x,
            y=est_y,
            mode='lines',
            name='Estimated',
            line=dict(color='blue', width=2, dash='solid'),
        ))
        
        # Start point
        fig.add_trace(go.Scatter(
            x=[0],
            y=[0],
            mode='markers',
            name='Start',
            marker=dict(color='green', size=12, symbol='circle', line=dict(color='darkgreen', width=2)),
        ))
        
        # End point - estimated
        fig.add_trace(go.Scatter(
            x=[est_x[-1]],
            y=[est_y[-1]],
            mode='markers',
            name='End (est)',
            marker=dict(color='blue', size=12, symbol='circle', line=dict(color='darkblue', width=2)),
        ))
        
        # End point - GT
        fig.add_trace(go.Scatter(
            x=[gt_x[-1]],
            y=[gt_y[-1]],
            mode='markers',
            name='End (GT)',
            marker=dict(color='green', size=12, symbol='x', line=dict(color='darkgreen', width=3)),
        ))
        
        # Compute final displacement error
        final_error = np.sqrt((est_x[-1] - gt_x[-1])**2 + (est_y[-1] - gt_y[-1])**2)
        
        title = (
            f"{seq_name}"
            f"<br>N={seq['N']}, pixel={seq['size_of_pixel']}m, step={seq['matching_step']} frames"
            f"<br>Pairs: {seq['num_pairs']}, Total frames: {seq['total_frames']}"
            f"<br>Avg rot error: {seq['avg_rot_error']:.2f} deg, Avg trans error: {seq['avg_trans_error']:.2f} m"
            f"<br>Final displacement error: {final_error:.2f} m"
        )
        
        fig.update_layout(
            title=title,
            height=700,
            width=900,
            xaxis_title='x (m)',
            yaxis_title='y (m)',
            legend=dict(x=0.01, y=0.99),
            margin=dict(l=60, r=60, t=80, b=60),
        )
        
        fig.update_yaxes(scaleanchor='x', scaleratio=1)
        
        return fig
    
else:
    print('No sequences found to plot.')

interactive(children=(IntSlider(value=0, continuous_update=False, description='Sequence', max=3), Output()), _…

In [6]:
if len(sequences) > 1:
    fig = go.Figure()
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
              '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    for i, seq in enumerate(sequences):
        seq_name = seq['name']
        traj = trajectories[seq_name]
        color = colors[i % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=traj['est_x'],
            y=traj['est_y'],
            mode='lines',
            name=seq_name,
            line=dict(color=color, width=2),
            opacity=0.8,
        ))
    
    # Add ground truth for first sequence as reference
    first_seq = sequences[0]
    first_traj = trajectories[first_seq['name']]
    fig.add_trace(go.Scatter(
        x=first_traj['gt_x'],
        y=first_traj['gt_y'],
        mode='lines',
        name=f'{first_seq["name"]} GT',
        line=dict(color='green', width=2, dash='dash'),
        opacity=0.6,
    ))
    
    fig.update_layout(
        title='All Estimated Trajectories Overlaid',
        height=700,
        width=900,
        xaxis_title='x (m)',
        yaxis_title='y (m)',
        legend=dict(x=0.01, y=0.99),
        margin=dict(l=60, r=60, t=80, b=60),
    )
    
    fig.update_yaxes(scaleanchor='x', scaleratio=1)
    fig.show()
else:
    print('Need at least 2 sequences for overlay plot.')

In [7]:
print('=== Benchmark Summary ===')
print()
for seq in sequences:
    seq_name = seq['name']
    print(f"--- {seq_name} ---")
    print(f"  N:               {seq['N']}")
    print(f"  Pixel size:      {seq['size_of_pixel']} m")
    print(f"  Matching step:   {seq['matching_step']} frames")
    print(f"  Pairs processed: {seq['num_pairs']}")
    print(f"  Total frames:    {seq['total_frames']}")
    print(f"  Avg rot error:   {seq['avg_rot_error']:.4f} deg")
    print(f"  Avg trans error: {seq['avg_trans_error']:.4f} m")
    print(f"  Avg confidence:  {seq['avg_confidence']:.2f}")
    print()
    
    # Compute trajectory span
    traj = trajectories[seq_name]
    est_span_x = traj['est_x'].max() - traj['est_x'].min()
    est_span_y = traj['est_y'].max() - traj['est_y'].min()
    gt_span_x = traj['gt_x'].max() - traj['gt_x'].min()
    gt_span_y = traj['gt_y'].max() - traj['gt_y'].min()
    final_error = np.sqrt((traj['est_x'][-1] - traj['gt_x'][-1])**2 + (traj['est_y'][-1] - traj['gt_y'][-1])**2)
    print(f"  Est trajectory:  {est_span_x:.1f} x {est_span_y:.1f} m")
    print(f"  GT trajectory:   {gt_span_x:.1f} x {gt_span_y:.1f} m")
    print(f"  Final error:     {final_error:.2f} m")
    print()

=== Benchmark Summary ===

--- seq00_fs2d_N128_p200_s5 ---
  N:               128
  Pixel size:      2.0 m
  Matching step:   5 frames
  Pairs processed: 828
  Total frames:    4142
  Avg rot error:   1.1178 deg
  Avg trans error: 1.4360 m
  Avg confidence:  24809.63

  Est trajectory:  2735.3 x 1086.0 m
  GT trajectory:   2297.1 x 1322.7 m
  Final error:     986.05 m

--- seq00_fs2d_N128_p50_s5 ---
  N:               128
  Pixel size:      0.5 m
  Matching step:   5 frames
  Pairs processed: 828
  Total frames:    4142
  Avg rot error:   19.9236 deg
  Avg trans error: 7.8917 m
  Avg confidence:  52613.74

  Est trajectory:  411.3 x 952.9 m
  GT trajectory:   2297.1 x 1322.7 m
  Final error:     147.81 m

--- seq01_fs2d_N128_p50_s5 ---
  N:               128
  Pixel size:      0.5 m
  Matching step:   5 frames
  Pairs processed: 769
  Total frames:    3848
  Avg rot error:   25.1083 deg
  Avg trans error: 10.2074 m
  Avg confidence:  49010.14

  Est trajectory:  1721.5 x 504.6 m
  GT t